In [29]:
# from langchain_ollama import ChatOllama

# llm = ChatOllama(model="qwen2.5:7b", temperature=0)
# response = llm.invoke("Hola soy Felipe y mi teléfono es +573001234567")
# response.pretty_print()


from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv

load_dotenv()
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite', google_api_key=os.getenv("GOOGLE_API_KEY"), temperature=0) 

In [30]:
from pydantic import BaseModel, Field

class ContactInfo(BaseModel):
    "Contact information for a person."

    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")
    tone: int = Field(description="The tone of the conversation (0 is informal, 100 is formal)", ge=0, le=100)
    age: int = Field(description="The age of the person", ge=1, le=120)
    sentiment: str = Field(description="The sentiment of the conversation, e.g., positive, negative, neutral.")

llm_with_structured_output = llm.with_structured_output(schema=ContactInfo)

messages = [
    ("system", """You are an expert at extracting structured contact information from conversations.

Your task:
- Extract all available contact details from the user's message
- Fill every field in the schema
- If a field is not mentioned, use null/None
- For 'tone': estimate formality from 0 (very informal) to 100 (very formal) based on writing style
- For 'sentiment': classify as positive, negative, or neutral based on the message mood
- For 'age': never invent an age, only extract if explicitly mentioned
- Always respond in the schema format, never as free text
"""),
    ("user", "Hola soy Felipe y mi teléfono es +573001234567")
]

response = llm_with_structured_output.invoke(messages)
response

ContactInfo(name='Felipe', email='null', phone='+573001234567', tone=20, age=25, sentiment='neutral')